# Lisbon hotel price history from stored snapshots

Two runs of
[Hotel Prices from Google Hotels: Vendor Rates, History](https://apify.com/yappman/google-hotels-prices):

1. **Tracked** mode, which charges only the start fee: the hotels and stays the Actor stores snapshots for.
2. **History** mode: the stored lead-price snapshots for up to five of those Lisbon hotels, for one stay. They are plotted as price per night against the date each snapshot was read.

**Cost at Apify Free-tier prices:** at most **$0.06** in total.

| Run | Events | Most it can cost |
|---|---|---|
| Tracked | $0.005 start only | $0.005 |
| History, up to 5 hotels | up to 5 x $0.01 rate history + $0.005 start | $0.055 |

More data: [data.yappman.com](https://data.yappman.com).

In [ ]:
import os
from datetime import timedelta
from decimal import Decimal
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from apify_client import ApifyClient

TOKEN = os.environ.get("APIFY_TOKEN")
if not TOKEN:
    raise RuntimeError("Set the APIFY_TOKEN environment variable first. See the README.")
client = ApifyClient(TOKEN)

OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)  # git-ignored: nothing you collect is committed


def run_actor(actor_id: str, run_input: dict, max_charge_usd: float):
    """Start an Actor, wait for it, and return the finished run.

    Spend is capped twice: maxItems in run_input limits the rows saved, and
    max_total_charge_usd stops the run if its charges would pass the cap.
    """
    run = client.actor(actor_id).call(
        run_input=run_input,
        max_total_charge_usd=Decimal(str(max_charge_usd)),
        run_timeout=timedelta(minutes=10),
    )
    if run is None or run.status != "SUCCEEDED":
        raise RuntimeError(f"{actor_id} did not succeed: {None if run is None else run.status}")
    print(f"{actor_id}: run {run.id}")
    print("Charged events:", run.charged_event_counts)
    return run

## 1. Find a stay with stored snapshots (tracked mode)

`tracked` mode writes its list to the `TRACKED_HOTELS` record of the run's key-value store, not to the dataset. Each item has `hotelId`, `checkIn`, `checkOut`, `adults` and `destination`. The code picks the Lisbon stay that has the most hotels.

In [ ]:
ACTOR = "yappman/google-hotels-prices"

tracked_run = run_actor(ACTOR, {"mode": "tracked"}, max_charge_usd=0.01)
record = client.key_value_store(tracked_run.default_key_value_store_id).get_record("TRACKED_HOTELS")
tracked = pd.DataFrame(record["value"]["items"])
lisbon = tracked[tracked["destination"].str.contains("Lisbon", case=False, na=False)]
# One stay (a checkIn/checkOut pair) with the most Lisbon hotels
stays = lisbon.groupby(["checkIn", "checkOut"])["hotelId"].nunique().sort_values(ascending=False)
print(len(lisbon), "Lisbon hotel-stays in the tracked list")
stays.head()

## 2. Read the history (paid)

`history` mode returns only stays stored for exactly the dates sent, for 2 adults and 1 room. It ignores `maxItems`, so the cost is capped by sending at most five hotel ids and by `max_total_charge_usd`. A hotel with only one snapshot is not charged for history.

In [ ]:
check_in, check_out = stays.index[0]
ids = list(lisbon[(lisbon["checkIn"] == check_in) & (lisbon["checkOut"] == check_out)]["hotelId"].head(5))
# Free tier: at most 5 x $0.01 + $0.005 = $0.055
hist_run = run_actor(
    ACTOR,
    {"mode": "history", "hotels": ids, "dates": {"checkIn": check_in, "checkOut": check_out}},
    max_charge_usd=0.06,
)
rows = [
    {"hotel": h["name"], "readAt": s["readAt"],
     "perNight": (s.get("leadPrice") or {}).get("perNight"), "deltaPerNight": s.get("deltaPerNight")}
    for h in client.dataset(hist_run.default_dataset_id).iterate_items() for s in h.get("history") or []
]
hist = pd.DataFrame(rows)
hist["readAt"] = pd.to_datetime(hist["readAt"])
hist.to_csv(OUTPUT / "lisbon_price_history.csv", index=False)
hist["snapshots"] = hist.groupby("hotel")["readAt"].transform("count")
hist.sort_values(["hotel", "readAt"])

## 3. Price per night over time

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for name, g in hist.groupby("hotel"):
    if len(g) < 2:
        continue  # a single snapshot is a point, not a trend
    ax.plot(g["readAt"], g["perNight"], marker="o", label=name[:30])
ax.set_ylabel("Lead price per night (USD)")
ax.set_title(f"Lisbon: stored lead-price snapshots, stay {check_in} to {check_out}")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()

## Notes

* Rows belong to the Actor's data terms, not to this repository's MIT licence. The CSV is written to `output/`, which git ignores.
* The tracked list can include stays whose dates have passed; the history is whatever the Actor stored for the stay you send.
* Example 05 reads the same history in an optional step, off by default.